# Escalated case: mix against rate

**Week 1, Tuesday afternoon. Notebook 4 of 5, the escalated case, unguided.** Round 3 ended with
the four-segment run in your own cell. This case takes the segment split to the next rung: whether
revenue per order rose because customers paid more, or because the mix of orders changed, and what
to say to Meera and the head of Retail-Plus about it.

> **The client asks.** "So revenue is customers, times how often they buy, times basket, times
> price. Now: which of those moved? Are we losing customers, or are the ones we have buying less?
> Marketing says more customers. Prove it or disprove it."
>
> Meera Raghavan, CEO, Kalpa Retail

> **And the tier asks.** "One of my members says the app's reorder button has been broken for six
> weeks. Is my tier the one slipping?"
>
> The head of Retail-Plus

> **Kavya's review of round 3.** "You rolled the segments up with their weights and got 1.65 and
> 1.25 back. Now use the same functions to split the fall, in orders and in rupees, and do not
> trust any helper you did not write until you have counted what it returns."

Each step has one or more `TODO` markers. Above each `__TODOn__` placeholder is a lettered choice;
replace the placeholder with the option you pick, run the cell, then run the check under it, which
tells you whether you were right without opening any solution. Run from the top: the notebook stops
at the first placeholder with a `NameError` naming it, which is intended.

**What to post:** the eleven letters in order as one string, then your sentence to Meera.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
ORDERS = kit.load_records("C2_W01_D02_orders_STUDENT.py")
SEGMENTS = ["Retail-Core", "Retail-Plus", "Business", "Student"]


def tree_for(rows):
    """The revenue tree's leaves for any list of orders, returned as a dictionary (round 3)."""
    revenue = 0
    seen = {}
    for order in rows:
        revenue += order["amount"]
        seen[order["customer_id"]] = True
    orders = len(rows)
    customers = len(seen)
    return {"revenue": revenue, "orders": orders, "customers": customers,
            "orders_per_customer": orders / customers,
            "revenue_per_order": revenue / orders}


def describe(amounts):
    """A group's typical value and spread: median, min, max and range (round 3)."""
    ordered = sorted(amounts)
    n = len(ordered)
    if n % 2 == 1:
        median = ordered[n // 2]
    else:
        median = (ordered[n // 2 - 1] + ordered[n // 2]) / 2
    return {"median": median, "min": ordered[0], "max": ordered[-1], "range": ordered[-1] - ordered[0]}


print(len(ORDERS), "orders loaded;", len(SEGMENTS), "segments")

In [ ]:
kit.side_by_side(
    kit.ladder(["Is the drop real?", "Which branch moved?", "Which segment?", "Escalated case: mix against rate", "Second case: the hypothesis"], lit=3, show=False),
    kit.vflow(["1. the tree per segment and quarter\nand a colleague's helper",
               "2. the orders bridge and the rupee bridge",
               "3. mix against rate\non revenue per order",
               "4. Retail-Plus orders per member",
               "5. the sentence to Meera\nwith two hypotheses"], show=False),
)

## Part 1. The tree for every segment in both quarters

Build `trees["Q1"][seg]` and `trees["Q2"][seg]` with `tree_for`, one list of orders per segment and
quarter.

In [ ]:
trees = {"Q1": {}, "Q2": {}}
for q in ("Q1", "Q2"):
    for seg in SEGMENTS:
        rows = []
        for order in ORDERS:
            # TODO 1. Which condition, beside the quarter, keeps one segment's orders?
            #   a) order["segment"] != seg
            #   b) order["segment"] == seg
            #   c) order["segment"] in SEGMENTS
            #   d) order["customer_id"] == seg
            if order["quarter"] == q and __TODO1__:
                rows.append(order)
        trees[q][seg] = tree_for(rows)
q1, q2 = trees["Q1"], trees["Q2"]

table_rows = []
for seg in SEGMENTS:
    table_rows.append((seg, f'{q1[seg]["customers"]}, {q2[seg]["customers"]}', f'{q1[seg]["orders"]}, {q2[seg]["orders"]}',
                       f'{q1[seg]["orders_per_customer"]:.2f}, {q2[seg]["orders_per_customer"]:.2f}',
                       f'{kit.rupees(q1[seg]["revenue"])}, {kit.rupees(q2[seg]["revenue"])}',
                       f'{kit.rupees(round(q1[seg]["revenue_per_order"]))}, {kit.rupees(round(q2[seg]["revenue_per_order"]))}'))
kit.table(["Segment", "Customers", "Orders", "Orders per customer", "Revenue", "Revenue per order"], table_rows,
          caption="Each cell reads Q1, then Q2")
opc_1, opc_2 = [], []
for seg in SEGMENTS:
    opc_1.append(round(q1[seg]["orders_per_customer"], 2))
    opc_2.append(round(q2[seg]["orders_per_customer"], 2))
kit.columns(SEGMENTS, [("Q1", opc_1), ("Q2", opc_2)], fmt=lambda v: f"{v:.2f}", lit=[1],
            title="Orders per customer by segment: the same members, half as often in Retail-Plus")

In [ ]:
customers_in = {"Q1": 0, "Q2": 0}
orders_in = {"Q1": 0, "Q2": 0}
for q in ("Q1", "Q2"):
    for seg in SEGMENTS:
        customers_in[q] += trees[q][seg]["customers"]
        orders_in[q] += trees[q][seg]["orders"]
kit.check("the segments' customers add back to 69 in each quarter", customers_in["Q1"] == customers_in["Q2"] == 69,
          f'{customers_in["Q1"]} and {customers_in["Q2"]}')
kit.check("the segments' orders add back to 114 and 86", (orders_in["Q1"], orders_in["Q2"]) == (114, 86),
          f'{orders_in["Q1"]} and {orders_in["Q2"]}')

A colleague left this helper and summary from last quarter's report. Run them exactly as they are,
then read what the summary says.

In [ ]:
def pct_change(before, after):
    change = 100 * (after - before) / before
    if abs(change) > 30:
        print(f"  check by hand: {change:+.1f}%")
    else:
        return round(change, 1)

changes = {}
for seg in SEGMENTS:
    changes[seg] = pct_change(q1[seg]["orders_per_customer"], q2[seg]["orders_per_customer"])
falls = {}
for seg, ch in changes.items():
    if ch and ch < 0:
        falls[seg] = ch
print("summary:", falls)
print("orders per customer fell in every segment, most in Business,", min(falls.values()), "percent")

Four segments went in. Count the rows that came out, and look at what `pct_change` hands back when
a change is larger than 30 percent.

In [ ]:
# TODO 2. Which test shows that some segments came back with no number at all?
#   a) None in changes.values()
#   b) len(changes) != len(SEGMENTS)
#   c) min(falls.values()) < -30
#   d) "Business" not in falls
lost = __TODO2__
kit.check("the summary's changes include segments with no number", lost is True)
kit.check("four segments in, two rows out", (len(SEGMENTS), len(falls)) == (4, 2), f"{len(SEGMENTS)} in, {len(falls)} out")

The summary would have told Meera to open Business accounts first, and the head of Retail-Plus that
his tier is not in the table. Fix the helper so it returns the change every time and puts the
"check by hand" flag in a column of its own.

In [ ]:
def pct_change_fixed(before, after):
    change = 100 * (after - before) / before
    # TODO 3. What should the fixed helper return?
    #   a) print(round(change, 1))
    #   b) round(change, 1) if abs(change) <= 30 else None
    #   c) round(change, 1)
    #   d) None
    return __TODO3__

fixed_rows = []
fixed = {}
for seg in SEGMENTS:
    fixed[seg] = pct_change_fixed(q1[seg]["orders_per_customer"], q2[seg]["orders_per_customer"])
    flag = "check by hand" if abs(fixed[seg]) > 30 else ""
    fixed_rows.append((seg, f'{q1[seg]["orders_per_customer"]:.2f}', f'{q2[seg]["orders_per_customer"]:.2f}', f"{fixed[seg]:+.1f}%", flag))
kit.table(["Segment", "Q1", "Q2", "Change", "Flag"], fixed_rows, caption="Orders per customer: every segment, every change")
kit.check("the fixed summary has a number for every segment", None not in fixed.values() and len(fixed) == 4)
kit.check("Retail-Plus fell 49.0 percent and Student rose 40.0", (fixed["Retail-Plus"], fixed["Student"]) == (-49.0, 40.0),
          f'{fixed["Retail-Plus"]} and {fixed["Student"]}')

## Part 2. The orders bridge and the rupee bridge

Twenty-eight orders were lost between the quarters. Put each segment's change on a bridge from 114
to 86.

In [ ]:
order_moves = {}
for seg in SEGMENTS:
    # TODO 4. Which expression is the segment's move on a bridge from Q1 to Q2?
    #   a) q1[seg]["orders"] - q2[seg]["orders"]
    #   b) q2[seg]["orders"] / q1[seg]["orders"]
    #   c) q2[seg]["orders_per_customer"] - q1[seg]["orders_per_customer"]
    #   d) q2[seg]["orders"] - q1[seg]["orders"]
    order_moves[seg] = __TODO4__

moves = []
for seg in SEGMENTS:
    moves.append((seg, order_moves[seg]))
kit.bridge(("Q1 orders", orders_in["Q1"]), moves, end_label="Q2 orders", lit=[1], fmt=lambda v: f"{v:,.0f}",
           title="Orders, 114 to 86, by segment: Retail-Plus is 25 of the 28")
lost_total = orders_in["Q1"] - orders_in["Q2"]
kit.check("the orders bridge lands on 86", orders_in["Q1"] + sum(order_moves.values()) == orders_in["Q2"])
kit.check("Retail-Plus is 25 of the 28 lost orders, 89 percent", order_moves["Retail-Plus"] == -25
          and round(-order_moves["Retail-Plus"] / lost_total * 100) == 89,
          f'{order_moves["Retail-Plus"]} of {-lost_total}')

Now the same bridge in rupees, from Rs 2,10,00,000 to Rs 1,87,00,000, and name the segment that
carries most of it.

In [ ]:
rupee_moves = {}
moves = []
for seg in SEGMENTS:
    rupee_moves[seg] = q2[seg]["revenue"] - q1[seg]["revenue"]
    moves.append((seg, rupee_moves[seg]))
revenue_q1 = 0
for seg in SEGMENTS:
    revenue_q1 += q1[seg]["revenue"]
kit.bridge(("Q1 revenue", revenue_q1), moves, end_label="Q2 revenue", lit=[2], lo=18000000,
           title="Revenue by segment; the axis starts at Rs 1.8 crore so the small moves show")

# TODO 5. Which segment carries most of the rupee fall?
#   a) "Retail-Plus"
#   b) "Retail-Core"
#   c) "Business"
#   d) "Student"
biggest_rupee_move = __TODO5__
kit.check("the named segment has the largest rupee fall", biggest_rupee_move == min(rupee_moves, key=rupee_moves.get),
          kit.rupees(rupee_moves[biggest_rupee_move]))
kit.check("the rupee bridge lands on Rs 1,87,00,000", revenue_q1 + sum(rupee_moves.values()) == 18700000)

Write one line under the bridge: how many Business orders carry that rupee fall, and what you would
check before anyone acts on it.

### Worked, not a TODO: the Business lens, described

Round 3's `describe` gives the median and the range. Add the middle half of the sorted orders, the
stretch between the first and third quartiles, and the mean with and without the single largest
order. Read them together: which of them does one order move?

In [ ]:
def quartile(ordered, fraction):
    """The value a fraction of the way through a sorted list, interpolating between neighbours."""
    position = (len(ordered) + 1) * fraction - 1
    low = int(position)
    return ordered[low] + (ordered[low + 1] - ordered[low]) * (position - low)

business = {}
for q, trees in (("Q1", q1), ("Q2", q2)):
    amounts = []
    for order in ORDERS:
        if order["quarter"] == q and order["segment"] == "Business":
            amounts.append(order["amount"])
    ordered = sorted(amounts)
    shape = describe(amounts)
    shape["middle half"] = quartile(ordered, 0.75) - quartile(ordered, 0.25)
    shape["mean"] = sum(ordered) / len(ordered)
    shape["mean without the largest"] = sum(ordered[:-1]) / (len(ordered) - 1)
    shape["amounts"] = ordered
    business[q] = shape

rows = []
for name in ("median", "middle half", "range", "mean", "mean without the largest"):
    change = (business["Q2"][name] / business["Q1"][name] - 1) * 100
    rows.append((name, kit.rupees(round(business["Q1"][name])), kit.rupees(round(business["Q2"][name])), f"{change:+.1f}%"))
kit.table(["Business, per order", "Q1", "Q2", "Change"], rows,
          caption="One order moves the range and the mean; the median and the middle half barely move")
kit.strip(business["Q2"]["amounts"],
          title=(f'Business orders in Q2: median {kit.rupees(round(business["Q2"]["median"]))}, '
                 f'mean {kit.rupees(round(business["Q2"]["mean"]))}; one order sits far to the right'))
kit.check("the Business middle half widened 13.1 percent, Rs 8,02,750 to Rs 9,08,000",
          (round(business["Q1"]["middle half"]), round(business["Q2"]["middle half"])) == (802750, 908000))
kit.check("without its largest order, Q2's Business mean falls below Q1's",
          business["Q2"]["mean without the largest"] < business["Q1"]["mean"],
          kit.rupees(round(business["Q2"]["mean without the largest"])))

**What it says.** The typical Business order barely moved: the median fell 3.2 percent and the
middle half widened 13.1 percent, while one order of Rs 29,45,460 stretched the range 72.6 percent
and lifted the mean Rs 1,15,924. The rupee lens is a count of three orders, each worth lakhs, which
is why it is checked before anyone acts on it.

## Part 3. Mix against rate on revenue per order

Revenue per order rose from Rs 1,84,211 to Rs 2,17,442, 18.0 percent. Either customers paid more
per order inside their segments (rate), or the orders that disappeared were the small ones (mix).
Price Q2's order mix at Q1's revenue per order in each segment, and compare.

In [ ]:
share_q1, share_q2 = {}, {}
for seg in SEGMENTS:
    share_q1[seg] = q1[seg]["orders"] / orders_in["Q1"]
    share_q2[seg] = q2[seg]["orders"] / orders_in["Q2"]

at_q2_mix = 0
for seg in SEGMENTS:
    # TODO 6. What adds up to revenue per order at Q2's mix and Q1's revenue per order?
    #   a) share_q1[seg] * q2[seg]["revenue_per_order"]
    #   b) share_q2[seg] * q1[seg]["revenue_per_order"]
    #   c) share_q2[seg] * q2[seg]["revenue_per_order"]
    #   d) (q1[seg]["revenue_per_order"] + q2[seg]["revenue_per_order"]) / 2
    at_q2_mix += __TODO6__

rpo_q1 = revenue_q1 / orders_in["Q1"]
rpo_q2 = (revenue_q1 + sum(rupee_moves.values())) / orders_in["Q2"]
mix = at_q2_mix - rpo_q1
rate = rpo_q2 - at_q2_mix
s1, s2 = [], []
for seg in SEGMENTS:
    s1.append(round(share_q1[seg] * 100, 1))
    s2.append(round(share_q2[seg] * 100, 1))
kit.columns(SEGMENTS, [("Q1 share of orders", s1), ("Q2 share of orders", s2)], fmt=lambda v: f"{v:.1f}%", lit=[1],
            title="The order mix: Retail-Plus fell from 44.7 to 30.2 percent of orders")
kit.bridge(("Q1 revenue per order", rpo_q1), [("mix: fewer small orders", mix), ("rate inside segments", rate)],
           end_label="Q2 revenue per order", lit=[0], lo=150000, fmt=lambda v: kit.rupees(round(v)),
           title="Revenue per order, split into mix and rate; the axis starts at Rs 1.5 lakh")
kit.check("at Q2's mix and Q1's rates, revenue per order is Rs 2,07,112", round(at_q2_mix) == 207112, kit.rupees(round(at_q2_mix)))
kit.check("mix and rate add back to the whole rise", abs(mix + rate - (rpo_q2 - rpo_q1)) < 1e-6,
          f"{kit.rupees(round(mix))} and {kit.rupees(round(rate))}")

In [ ]:
# TODO 7. What share of the rise in revenue per order does the mix explain?
#   a) mix / rate
#   b) mix / rpo_q2
#   c) mix / (rpo_q2 - rpo_q1)
#   d) rate / (rpo_q2 - rpo_q1)
mix_share = __TODO7__
print(f"mix explains {mix_share * 100:.0f} percent of the rise in revenue per order")
kit.check("mix explains about 69 percent of the rise", round(mix_share * 100) == 69, f"{mix_share * 100:.1f}")

Write one line: did Kalpa's customers pay more per order, and what share of the rise says so?

### Worked, not a TODO: move the rate first

The split above moved the mix first, pricing Q2's mix at Q1's rates. Move the rate first instead,
pricing Q1's mix at Q2's rates, and the same Rs 33,231 splits a little differently, because the
part where mix and rate moved together is charged to whichever goes second. The morning's bridge
had the same property.

In [ ]:
at_q1_mix_q2_rates = 0
for seg in SEGMENTS:
    at_q1_mix_q2_rates += share_q1[seg] * q2[seg]["revenue_per_order"]
rate_first = at_q1_mix_q2_rates - rpo_q1
mix_second = rpo_q2 - at_q1_mix_q2_rates
rise = rpo_q2 - rpo_q1

kit.table(["Order of the steps", "Mix", "Rate", "Mix share"],
          [("mix first", kit.rupees(round(mix)), kit.rupees(round(rate)), f"{mix / rise * 100:.1f}%"),
           ("rate first", kit.rupees(round(mix_second)), kit.rupees(round(rate_first)), f"{mix_second / rise * 100:.1f}%")],
          caption="Revenue per order, Rs 1,84,211 to Rs 2,17,442, split in both orders")
kit.columns(["mix first", "rate first"], [("mix", [mix, mix_second]), ("rate", [rate, rate_first])],
            fmt=lambda v: kit.rupees(round(v)), width=560,
            title="Either order, the mix carries about seven tenths of the rise")
kit.check("moved second, the mix explains Rs 24,028", round(mix_second) == 24028, kit.rupees(round(mix_second)))
kit.check("in both orders the mix explains more than two thirds of the rise",
          mix / rise > 2 / 3 and mix_second / rise > 2 / 3, f"{mix / rise * 100:.1f} and {mix_second / rise * 100:.1f}")

**What it says.** Mix explains 69 percent of the rise moved first and 72 percent moved second; the
answer, that nobody had to pay more, does not depend on the order. Say which order you used anyway,
so the reader can rebuild the split.

## Part 4. Retail-Plus orders per member

The tier's total fell by half. Find out whether a few members stopped or every member slowed down,
by counting how many members ordered once, twice or three times in each quarter.

In [ ]:
per_member = {"Q1": {}, "Q2": {}}
plus_amounts = {"Q1": [], "Q2": []}
for order in ORDERS:
    if order["segment"] == "Retail-Plus":
        q, cid = order["quarter"], order["customer_id"]
        per_member[q][cid] = per_member[q].get(cid, 0) + 1
        plus_amounts[q].append(order["amount"])

shape = {"Q1": {}, "Q2": {}}
for q in ("Q1", "Q2"):
    for cid in per_member[q]:
        n = per_member[q][cid]
        # TODO 8. How does shape[q] count the members who placed n orders?
        #   a) shape[q].get(n, 0) + 1
        #   b) shape[q][n] + 1
        #   c) n + 1
        #   d) shape[q].get(cid, 0) + n
        shape[q][n] = __TODO8__

kit.columns(["once", "twice", "three times"],
            [("Q1 members", [shape["Q1"].get(1, 0), shape["Q1"].get(2, 0), shape["Q1"].get(3, 0)]),
             ("Q2 members", [shape["Q2"].get(1, 0), shape["Q2"].get(2, 0), shape["Q2"].get(3, 0)])],
            title="Retail-Plus members by how many orders each placed")
plus_shape = {"Q1": describe(plus_amounts["Q1"]), "Q2": describe(plus_amounts["Q2"])}
kit.table(["Retail-Plus", "Members", "Orders", "Median order", "Min", "Max"],
          [("Q1", len(per_member["Q1"]), len(plus_amounts["Q1"]), kit.rupees(plus_shape["Q1"]["median"]),
            kit.rupees(plus_shape["Q1"]["min"]), kit.rupees(plus_shape["Q1"]["max"])),
           ("Q2", len(per_member["Q2"]), len(plus_amounts["Q2"]), kit.rupees(plus_shape["Q2"]["median"]),
            kit.rupees(plus_shape["Q2"]["min"]), kit.rupees(plus_shape["Q2"]["max"]))],
          caption="The same members, fewer and slightly larger orders")
kit.check("22 members bought in each quarter", len(per_member["Q1"]) == len(per_member["Q2"]) == 22)
kit.check("the shape adds back to the members", sum(shape["Q1"].values()) == 22 and sum(shape["Q2"].values()) == 22)
kit.check("no member ordered three times in Q2", shape["Q2"].get(3, 0) == 0)

Describe the shape in two sentences: did members leave, or did the same members order less often?

## Part 5. The sentence to Meera, with two hypotheses

The sentence leads with the behaviour finding, names the rupee finding with its caveat, and names
each cause as a hypothesis with the evidence that would settle it.

In [ ]:
EVIDENCE = {
    "export": "orders per customer by segment from this export, again",
    "campaigns": "Marketing's new-customer campaign reach by month",
    "app_logs": "the app's reorder events and failures by week, and the release that broke it",
    "tier_log": "the tier's change log for benefits, prices and delivery terms, with renewals and support tickets",
}

# TODO 9. Which segment leads the sentence as the behaviour finding?
#   a) "Business"
#   b) "Retail-Plus"
#   c) "Student"
#   d) "all four segments"
lead_behaviour = __TODO9__

# TODO 10. Which evidence settles hypothesis 1, the broken reorder feature?
#   a) "export"
#   b) "campaigns"
#   c) "tier_log"
#   d) "app_logs"
settles_h1 = EVIDENCE[__TODO10__]

# TODO 11. Which evidence settles hypothesis 2, something that changed for members in July?
#   a) "tier_log"
#   b) "export"
#   c) "app_logs"
#   d) "campaigns"
settles_h2 = EVIDENCE[__TODO11__]

kit.check("the behaviour finding names the segment that lost the most orders",
          lead_behaviour == min(order_moves, key=order_moves.get))
kit.check("both hypotheses need data this export does not carry",
          EVIDENCE["export"] not in (settles_h1, settles_h2) and EVIDENCE["campaigns"] not in (settles_h1, settles_h2))
kit.check("the two hypotheses are settled by different evidence", settles_h1 != settles_h2)
kit.flow([f'behaviour\n{lead_behaviour}: {q1[lead_behaviour]["orders"]} to {q2[lead_behaviour]["orders"]} orders',
          f'rupees\nBusiness, on {-order_moves["Business"]} orders',
          "H1 reorder feature\napp logs by week", "H2 a July change\ntier change log"],
         kinds=["bad", "unknown", "known", "known"], title="The four parts of the sentence to Meera")

**A rival to both hypotheses: the season.** Q2 is the monsoon quarter, and a seasonal dip would also
begin in July. Retail-Core stayed near flat through the same months, which argues against a season
that hit every customer, but a season could still hit a paid tier harder. Last year's Q2 for
Retail-Plus against Retail-Core settles it, and it goes on the same data request.

**Before trusting the app's logs, ask how they are written.** A failure log that records only the
attempts that reached the server misses every member whose tap never got that far, and that absence
is not random: it is the morning's missing discount field in a new place.

### Your sentence to Meera

Write it in this order: the closed-quarter fall with its windows; the branch that moved and the
one that did not; the behaviour finding with its count; the rupee finding with its caveat; and two
hypotheses, each with the evidence that would settle it.

*Write your sentence here.*

### In the interview

**[F] Revenue per order rose 18 percent while revenue fell; did prices go up?**

**[D] The fall is in rupees in one segment and in behaviour in another; which do you put in front
of the CEO first?**

Answer both aloud in under a minute each, using the numbers from Parts 2 and 3.

In [ ]:
kit.check_summary()
print("Post your eleven letters in order as one string, then your sentence to Meera.")